# Prediction of Product Sales

- Author: Salam Odeh

## Preprocessing

**Important - Avoiding Data Leakage:** in Parts 2-4 we cleaned the data and filled missing values with simple placeholders *before* doing any train/test split. That approach is fine for exploratory analysis, but it is NOT appropriate for modeling, because:
- Any imputation statistic (like a median or mode) calculated on the FULL dataset would "leak" information from the test set into the training process.
- Best practice is to perform the train/test split FIRST, and then fit any imputers/scalers/encoders ONLY on the training data.

Therefore, in this notebook we will load a **fresh, unmodified copy** of the original dataset and restart our cleaning process correctly, with imputation occurring after the split.

In [1]:
# Mount Google Drive (Colab specific - skip if running locally)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

from sklearn import set_config
set_config(transform_output='pandas')  # keep outputs as DataFrames

pd.set_option('display.max_columns', 100)

### Load a Fresh Version of the Original Dataset

In [3]:
# Load a FRESH copy of the original, uncleaned dataset using pd.read_csv()
# Adjust this path to wherever you saved the CSV in your Drive
fpath =  '/content/drive/MyDrive/Colab Notebooks/0.0 - project/Sales Prediction/sales_predictions_2023.csv'

df = pd.read_csv(fpath)
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [4]:
# Confirm we are starting fresh - nulls should still be present
# and Item_Fat_Content should still have its inconsistent categories
print(df.isna().sum())
print()
print(df['Item_Fat_Content'].unique())

Item_Identifier                 0
Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

['Low Fat' 'Regular' 'low fat' 'LF' 'reg']


### Deal with Duplicates and Inconsistent Categorical Data (Before the Split)

Per the instructions, duplicates and inconsistent categorical labels should be handled BEFORE splitting the data (these are data quality issues, not statistics that could leak information - fixing a typo like `'low fat'` -> `'Low Fat'` does not use any information from the target or about train vs. test rows).

In [5]:
# Check for and drop any duplicate rows
print(f"Duplicate rows found: {df.duplicated().sum()}")
df = df.drop_duplicates()

Duplicate rows found: 0


In [6]:
# Fix inconsistent categories in Item_Fat_Content
fat_content_map = {
    'low fat': 'Low Fat',
    'LF': 'Low Fat',
    'reg': 'Regular'
}
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_content_map)

# Confirm only 2 consistent categories remain
df['Item_Fat_Content'].value_counts()

,count
Item_Fat_Content,
Low Fat,5517
Regular,3006


### Identify Features (X) and Target (y)

In [7]:
# Target: Item_Outlet_Sales
y = df['Item_Outlet_Sales']

# Features: drop the target AND Item_Identifier
# (Item_Identifier has very high cardinality - 1,559 unique product IDs -
#  with no real predictive meaning, as discussed in Part 4's Feature
#  Inspection. We drop it here to avoid creating 1,559 one-hot columns.)
X = df.drop(columns=['Item_Outlet_Sales', 'Item_Identifier'])

X.head()

,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type
0,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1
1,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2
2,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1
3,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store
4,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1


### Perform the Train-Test Split

In [8]:
# Split the data BEFORE any imputation, to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Training: {X_train.shape}")
print(f"Testing:  {X_test.shape}")

Training: (6392, 10)
Testing:  (2131, 10)


### Confirm Missing Values (on Training Data Only)

We only inspect/fit on the training data from this point forward, to make sure no information from the test set influences our preprocessing decisions.

In [9]:
# Check for missing values in the training features
X_train.isna().sum()

,0
Item_Weight,1107
Item_Fat_Content,0
Item_Visibility,0
Item_Type,0
Item_MRP,0
Outlet_Identifier,0
Outlet_Establishment_Year,0
Outlet_Size,1812
Outlet_Location_Type,0
Outlet_Type,0


- `Item_Weight` and `Outlet_Size` have missing values, consistent with what we found during EDA in Parts 2 and 4. These will be handled by `SimpleImputer` inside our preprocessing pipeline below (imputation statistics like the median will be calculated using ONLY the training data).

## Create a Preprocessing Object

### Preprocessing Pipeline for Numerical Features

Our numeric pipeline will fill in missing values with the **median** (using `SimpleImputer`), then scale the features with `StandardScaler`.

In [10]:
# Save list of numeric columns
num_cols = X_train.select_dtypes('number').columns
num_cols

Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year'],
      dtype='object')

In [11]:
# Constructing numeric preprocessing objects
num_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

num_pipe = make_pipeline(num_imputer, scaler)
num_tuple = ('num', num_pipe, num_cols)
num_tuple

('num',
 Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                 ('standardscaler', StandardScaler())]),
 Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
        'Outlet_Establishment_Year'],
       dtype='object'))

### Preprocessing Pipeline for Categorical Features

Our categorical pipeline will fill in missing values with the placeholder `'MISSING'` (using `SimpleImputer`), then one-hot encode the categorical features.

In [12]:
# Save list of categorical columns
cat_cols = X_train.select_dtypes('object').columns
cat_cols

Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size',
       'Outlet_Location_Type', 'Outlet_Type'],
      dtype='object')

In [13]:
# Constructing categorical preprocessing objects
cat_imputer = SimpleImputer(strategy='constant', fill_value='MISSING')
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cat_pipe = make_pipeline(cat_imputer, cat_encoder)
cat_tuple = ('cat', cat_pipe, cat_cols)
cat_tuple

('cat',
 Pipeline(steps=[('simpleimputer',
                  SimpleImputer(fill_value='MISSING', strategy='constant')),
                 ('onehotencoder',
                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
 Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size',
        'Outlet_Location_Type', 'Outlet_Type'],
       dtype='object'))

### Combine into a Column Transformer

In [14]:
# Define a column transformer that applies the appropriate pipeline
# to numeric vs. categorical columns
preprocessor = ColumnTransformer(
    [num_tuple, cat_tuple],
    verbose_feature_names_out=False
)
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['Item_Weight', 'Item_Visibility', 'Item_MRP',
       'Outlet_Establishment_Year'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='MISSING',
                                                                strategy='constant')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size',
       'Outlet_Location_Type', 'Outlet_Type'],
      dtype='object'))],
                  verbose_feature_names_out=False)

### Fit the Preprocessor on Training Data and Transform Both Sets

We `fit_transform` on the **training data only**, then use the already-fitted preprocessor to `transform` the **test data**. This ensures all imputation statistics (medians) and encoding categories are learned exclusively from the training set.

In [15]:
# Fit on training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

# Transform the test data using the already-fitted preprocessor
X_test_processed = preprocessor.transform(X_test)

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed test shape:     {X_test_processed.shape}")

X_train_processed.head()

Processed training shape: (6392, 43)
Processed test shape:     (2131, 43)


,Item_Weight,Item_Visibility,Item_MRP,Outlet_Establishment_Year,Item_Fat_Content_Low Fat,Item_Fat_Content_Regular,Item_Type_Baking Goods,Item_Type_Breads,Item_Type_Breakfast,Item_Type_Canned,Item_Type_Dairy,Item_Type_Frozen Foods,Item_Type_Fruits and Vegetables,Item_Type_Hard Drinks,Item_Type_Health and Hygiene,Item_Type_Household,Item_Type_Meat,Item_Type_Others,Item_Type_Seafood,Item_Type_Snack Foods,Item_Type_Soft Drinks,Item_Type_Starchy Foods,Outlet_Identifier_OUT010,Outlet_Identifier_OUT013,Outlet_Identifier_OUT017,Outlet_Identifier_OUT018,Outlet_Identifier_OUT019,Outlet_Identifier_OUT027,Outlet_Identifier_OUT035,Outlet_Identifier_OUT045,Outlet_Identifier_OUT046,Outlet_Identifier_OUT049,Outlet_Size_High,Outlet_Size_MISSING,Outlet_Size_Medium,Outlet_Size_Small,Outlet_Location_Type_Tier 1,Outlet_Location_Type_Tier 2,Outlet_Location_Type_Tier 3,Outlet_Type_Grocery Store,Outlet_Type_Supermarket Type1,Outlet_Type_Supermarket Type2,Outlet_Type_Supermarket Type3
4776,0.827485,-0.712775,1.828109,1.327849,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
7510,0.566644,-1.291052,0.603369,1.327849,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
5828,-0.121028,1.813319,0.244541,0.136187,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
5327,-1.158464,-1.004931,-0.952591,0.732018,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4810,1.538870,-0.965484,-0.336460,0.493686,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


## Summary

In this notebook, we:
1. Reloaded the **original, fresh** dataset to avoid any data leakage from our earlier exploratory cleaning.
2. Dropped duplicates and fixed inconsistent `Item_Fat_Content` categories (safe to do before the split, since these are simple data-quality fixes).
3. Defined `X` (features, dropping `Item_Identifier` due to high cardinality) and `y` (target = `Item_Outlet_Sales`).
4. Performed the **train-test split FIRST**, before any imputation.
5. Built a `ColumnTransformer` preprocessing object:
   - Numeric features: median imputation -> `StandardScaler`
   - Categorical features: `'MISSING'` placeholder imputation -> `OneHotEncoder`
6. Fit the preprocessor on the training data only, then used it to transform both the training and test sets.

We will finalize this project in Part 6, where we use `X_train_processed`, `X_test_processed`, `y_train`, and `y_test` (or a full model pipeline including the `preprocessor`) to build and evaluate Linear Regression and Random Forest models.